# Train the ML signal model

Logistic regression on the last 5 5-minute returns of BTCUSDT.
Target = sign of the next 5-minute return (1 if up, 0 otherwise).

Run all cells. Outputs `model.pkl` next to this notebook, which
`ml_based_strategy.py` and `ml_based_strategy_ws.py` will load.

In [1]:
import pickle
from pathlib import Path

import polars as pl
from deltalake import DeltaTable
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

from quantlake.config import SILVER_ROOT
from quantlake.gold.ohlcv import resample

SYMBOL       = "BTCUSDT"
SILVER_TABLE = "binance_spot_ohlcv"
TIMEFRAME    = "5m"
N_FEATURES   = 5  # number of past returns used as features
MODEL_PATH   = Path("model.pkl")

In [2]:
# Load 1m silver, resample to 5m
path = str(SILVER_ROOT / SILVER_TABLE)
df = (
    pl.from_arrow(DeltaTable(path).to_pyarrow_table(filters=[("symbol", "=", SYMBOL)]))
    .sort("timestamp")
)
df = resample(df, TIMEFRAME).select("timestamp", "close").drop_nulls()
print(f"{len(df):,} 5m bars")
df.tail()

17,377 5m bars


timestamp,close
"datetime[μs, UTC]",f64
2026-05-02 07:40:00 UTC,78227.11
2026-05-02 07:45:00 UTC,78218.04
2026-05-02 07:50:00 UTC,78234.05
2026-05-02 07:55:00 UTC,78212.07
2026-05-02 08:00:00 UTC,78175.03


In [3]:
# Build features (last N returns) + target (next return sign)
df = df.with_columns(
    pl.col("close").pct_change().alias("ret"),
).drop_nulls()

for i in range(N_FEATURES):
    df = df.with_columns(pl.col("ret").shift(i).alias(f"f{i}"))

df = df.with_columns(
    (pl.col("close").shift(-1) > pl.col("close")).cast(pl.Int8).alias("target"),
).drop_nulls()

X = df.select([f"f{i}" for i in range(N_FEATURES)]).to_numpy()
y = df["target"].to_numpy()
print(f"X={X.shape}  y={y.shape}  positive rate={y.mean():.3f}")

X=(17371, 5)  y=(17371,)  positive rate=0.499


In [4]:
# Time-ordered split (no shuffle) - never train on future data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print(f"train accuracy: {accuracy_score(y_train, model.predict(X_train)):.3f}")
print(f"test  accuracy: {accuracy_score(y_test,  model.predict(X_test)):.3f}")

train accuracy: 0.500
test  accuracy: 0.504


In [5]:
with open(MODEL_PATH, "wb") as f:
    pickle.dump(model, f)
print(f"saved -> {MODEL_PATH.resolve()}")

saved -> /Users/lucasinglese/Desktop/quantlake/applications/live/ml_based_strategy/model.pkl
